# 01 — Kilian structural VAR shocks

Milestone A. Thin consumer of `src.var_shocks`. Fits the 24-lag Cholesky-identified VAR on `[dprod, kilian_rea, real_oil_price_diff]`, extracts the three structural shock series, validates against Kilian's published decomposition, and inspects the largest shock episodes per channel.

**All logic lives in `src/`; this notebook is for exploration and sanity-checking only.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.io_utils import load_panel
from src.var_shocks import fit_kilian_var, extract_shocks, validate_shocks, plot_historical_decomposition

panel = load_panel()
panel.shape, panel.index.min().date(), panel.index.max().date()

## Fit VAR and extract shocks

In [ ]:
results = fit_kilian_var(panel)
print(f'VAR: k={results.neqs} eqs, {results.k_ar} lags, {results.nobs} usable obs')
print(f'AIC={results.aic:.3f}  BIC={results.bic:.3f}  log-lik={results.llf:.1f}')

In [ ]:
shocks = extract_shocks(results)
shocks.describe().round(3)

In [ ]:
# Orthogonality check — Cholesky should make shocks uncorrelated
shocks.corr().round(3)

## Validation against Kilian's published pattern

In [ ]:
validate_shocks(shocks)

## Largest shocks per channel

Sanity check: these should line up with known oil-market events.

In [ ]:
def top_events(col, n=10):
    top = shocks[col].abs().nlargest(n).index
    return shocks.loc[top, col].round(2).to_frame(f'{col} z-score').sort_values(f'{col} z-score', key=abs, ascending=False)

for c in ['eps_supply', 'eps_agg_demand', 'eps_precaut']:
    print(f'\n=== Top |{c}| events ===')
    print(top_events(c, 10).to_string())

## Historical decomposition plot

Regenerated inline for visual inspection — matches `outputs/report/figures/historical_decomposition.png`.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col in zip(axes, ['eps_supply', 'eps_agg_demand', 'eps_precaut']):
    ax.axhline(0, color='black', lw=0.5)
    ax.axhline(1.5, color='red', lw=0.4, ls='--', alpha=0.5)
    ax.axhline(-1.5, color='red', lw=0.4, ls='--', alpha=0.5)
    ax.plot(shocks.index, shocks[col], lw=0.9, color='#1f3b66')
    ax.fill_between(shocks.index, 0, shocks[col], alpha=0.25, color='#1f3b66')
    ax.set_title(col, loc='left')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()